In [0]:
%sql
create table if not exists gizmobox_gr.delta_lake.raw_stock_prices

In [0]:
%sql
copy into gizmobox_gr.delta_lake.raw_stock_prices
from 'abfs://gizmoboxgr@ourdbpracticeextdl.dfs.core.windows.net/landing/stock_prices'
fileformat = JSON
format_options ('inferSchema' = 'true')
copy_options('mergeSchema' = 'true')

In [0]:
%sql
select * from gizmobox_gr.delta_lake.raw_stock_prices

**Merge The Source Data Into Target**

In [0]:
%sql
create table if not exists gizmobox_gr.delta_lake.stock_prices
(
    stock_id string,
    trading_date string,
    price double
)

In [0]:
%sql

-- delete from gizmobox_gr.delta_lake.stock_prices
merge into gizmobox_gr.delta_lake.stock_prices as target
using gizmobox_gr.delta_lake.raw_stock_prices as source
on target.stock_id = source.stock_id
when matched and source.status='ACTIVE' then
    update set target.price=source.price,target.trading_date=source.trading_date
when matched and source.status='DELISTED' then
    delete
when not matched and source.status='ACTIVE' then
    insert (stock_id,trading_date,price) values (source.stock_id,source.trading_date, source.price)


In [0]:
%sql
select * from gizmobox_gr.delta_lake.stock_prices;